# Nike Shoe Sales Analysis

## Business Question
What drives perceived value and consumer demand in Nike's product catalog?

## Key Questions
1. Which shoe lines have the highest average sale price?
2. Which shoe lines have the highest average rating (rated products only)?
3. What is the relationship between price tier and average rating?
4. Which individual products have the most reviews?
5. How does the presence vs. absence of a listing price affect average rating and reviews?
6. What percentage of products fall into each shoe line?

## Data Set
    Nike Shoe Sales
        Rows:643
        Columns: 10

## Tools
    - Python
    - SQL (SQLite)
    - Tableau

## Phase 1: Data Cleaning

In [2]:
#Data Manipulation
import pandas as pd
import numpy as np

#Visualization
import matplotlib.pyplot as plt
import seaborn as sns

#Read CSV file into a DataFrame
df = pd.read_csv(r'C:\Users\JaceCordell\Desktop\nike_shoe_analysis\nike_shoes_sales_data.csv')

In [3]:
#Get a brief overview of the data
print('SHAPE OF DF: \n', df.shape)
print('TYPES IN DF: \n', df.dtypes)
print('TOP OF DF: \n', df.head())

SHAPE OF DF: 
 (643, 10)
TYPES IN DF: 
 product_name      object
product_id        object
listing_price      int64
sale_price         int64
discount           int64
brand             object
description       object
rating           float64
reviews            int64
images            object
dtype: object
TOP OF DF: 
                      product_name  product_id  listing_price  sale_price  \
0  Nike Air Force 1 '07 Essential  CJ1646-600              0        7495   
1            Nike Air Force 1 '07  CT4328-101              0        7495   
2    Nike Air Force 1 Sage Low LX  CI3482-200              0        9995   
3             Nike Air Max Dia SE  CD0479-200              0        9995   
4             Nike Air Max Verona  CZ6156-101              0        9995   

   discount brand                                        description  rating  \
0         0  Nike  Let your shoe game shimmer in the Nike Air For...     0.0   
1         0  Nike  The legend lives on in the Nike Air Force 1 '0.

In [4]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 643 entries, 0 to 642
Data columns (total 10 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   product_name   643 non-null    object 
 1   product_id     643 non-null    object 
 2   listing_price  643 non-null    int64  
 3   sale_price     643 non-null    int64  
 4   discount       643 non-null    int64  
 5   brand          643 non-null    object 
 6   description    640 non-null    object 
 7   rating         643 non-null    float64
 8   reviews        643 non-null    int64  
 9   images         572 non-null    object 
dtypes: float64(1), int64(4), object(5)
memory usage: 50.4+ KB


The dataset contains 643 records and 10 columns. Notable nulls: 3 in `description` and 71 in `images` (~11%). Pricing columns (`listing_price`, `sale_price`) are stored as integers and will need to be scaled to USD. The `rating` column shows 0.0 for unrated products — these are sentinel values, not actual scores.

In [5]:
print(df['reviews'].value_counts())
print(df['reviews'].value_counts(normalize=True).round(3))

reviews
0      229
1       99
2       51
3       35
5       31
4       21
6       17
10      11
8       10
18       9
7        9
12       9
9        9
11       8
13       7
19       7
34       7
20       6
17       6
25       4
23       4
15       4
31       4
14       4
30       3
45       3
24       3
47       2
26       2
61       2
63       2
16       2
27       2
33       2
36       1
62       1
72       1
71       1
78       1
29       1
42       1
65       1
94       1
105      1
68       1
54       1
50       1
22       1
223      1
81       1
67       1
48       1
53       1
Name: count, dtype: int64
reviews
0      0.356
1      0.154
2      0.079
3      0.054
5      0.048
4      0.033
6      0.026
10     0.017
8      0.016
18     0.014
7      0.014
12     0.014
9      0.014
11     0.012
13     0.011
19     0.011
34     0.011
20     0.009
17     0.009
25     0.006
23     0.006
15     0.006
31     0.006
14     0.006
30     0.005
45     0.005
24     0.005
47     0.003
26     0.00

In [6]:
print(df.isnull().sum())

product_name      0
product_id        0
listing_price     0
sale_price        0
discount          0
brand             0
description       3
rating            0
reviews           0
images           71
dtype: int64


No nulls remain in the analytical columns. `description` and `images` will be dropped as they have no analytical value for this project's key questions.

In [7]:
print(df.duplicated().sum())

88


In [8]:
print(df[df.duplicated(keep=False)].sort_values('product_name').head(20))

             product_name  product_id  listing_price  sale_price  discount  \
492      Air Jordan 1 Low  553558-125              0        8995         0   
189      Air Jordan 1 Low  553558-125              0        8995         0   
126   Air Jordan 12 Retro  130690-017              0       15995         0   
437   Air Jordan 12 Retro  130690-017              0       15995         0   
561   Air Jordan 13 Retro  414571-030              0       15995         0   
322   Air Jordan 13 Retro  414571-030              0       15995         0   
603    Air Jordan 5 Retro  CN2932-100              0       18995         0   
344    Air Jordan 5 Retro  CN2932-100              0       18995         0   
608    Air Jordan 7 Retro  313358-006          15995       11197         0   
537    Air Jordan 7 Retro  313358-006          15995       11197         0   
208         Air Jordan OG  133000-106              0       11995         0   
499         Air Jordan OG  133000-106              0       11995

In [9]:
df = df.drop_duplicates()
print(df.shape)

(555, 10)


# Duplicate Removal
88 duplicate rows identified and removed (13.7% of raw records). Likely caused by product re-listings or scraping overlap. Final dataset: 555 records.

In [10]:
print(df['discount'].value_counts())

discount
0    555
Name: count, dtype: int64


`discount` is 0 for all 555 remaining records and carries no analytical value. It will be dropped along with `description` and `images`.

In [11]:
#Drop discount (all 0)
#Drop images and description. They have no analytical value
df = df.drop(columns=['discount','images','description'])
print(df.shape)

(555, 7)


In [16]:
#Create flags for whether or not item has rating and listing price. One flag for each
df['is_rated'] = df['rating'] > 0.0
print(df['is_rated'].value_counts())

df['has_msrp'] = df['listing_price'] > 0.0
print(df['has_msrp'].value_counts())

is_rated
True     365
False    190
Name: count, dtype: int64
has_msrp
False    362
True     193
Name: count, dtype: int64


Two boolean flags created: `is_rated` identifies the 365 products (65.8%) with a consumer rating above 0.0. `has_msrp` identifies the 193 products (34.8%) with an original listing price. These flags will be used to scope SQL queries for questions 2, 3, and 5.

In [51]:
# Streamline long product names into standardized, single-word/phrase shoe line categories

def get_shoe_line(name):
    name = str(name)
    if 'Air Force'    in name: return 'Air Force'
    elif 'AF-1'       in name: return 'Air Force'    
    elif 'Air Max'    in name: return 'Air Max'
    elif 'Pegasus'    in name: return 'Pegasus'
    elif 'Zoom'       in name: return 'Zoom'
    elif 'Jordan'     in name: return 'Jordan'
    elif 'React'      in name: return 'React'
    elif 'Metcon'     in name: return 'Metcon'
    elif 'Mercurial'  in name: return 'Mercurial'
    elif 'Free'       in name: return 'Free'
    elif 'Blazer'     in name: return 'Blazer'
    elif 'Flyknit'    in name: return 'Flyknit'
    elif 'Court'      in name: return 'Court'
    elif 'Phantom'    in name: return 'Phantom'
    elif 'Renew'      in name: return 'Renew'
    elif any(player in name for player in ['KD','Kyrie','LeBron','PG']): return 'NBA'
    elif 'VaporMax'   in name: return 'VaporMax'
    elif 'Flex'       in name: return 'Flex'
    elif 'React'      in name: return 'React'
    elif 'Renew'      in name: return 'Renew'
    elif 'Benassi'    in name: return 'Benassi'
    elif 'Joyride'    in name: return 'Joyride'
    elif 'SB'         in name: return 'SB'
    elif 'Tiempo'     in name: return 'Tiempo'
    elif 'Air'        in name: return 'Air'
    elif 'MX'         in name: return 'MX'
    else:                      return 'Other'

df['shoe_line']=df['product_name'].apply(get_shoe_line)
print(df['shoe_line'].value_counts())

shoe_line
Air Max      135
Zoom          52
Other         47
React         43
Jordan        39
Mercurial     33
Air Force     29
NBA           26
Pegasus       23
Phantom       14
Metcon        13
Free          13
Air           12
VaporMax      10
Joyride        9
Tiempo         9
SB             8
Flex           7
Flyknit        6
Court          6
Blazer         6
Benassi        6
MX             5
Renew          4
Name: count, dtype: int64


Product names were parsed into 24 shoe line categories using keyword matching on the `product_name` field. Air Max dominates the catalog at 135 products. The function checks for more specific terms first (e.g., "Air Force" before "Air") to prevent misclassification.

In [52]:
print(df.loc[df['shoe_line'] == 'Other', 'product_name'].sort_values())

115                 Nike ACG MOC 3.0
95                    Nike AlphaDunk
234                   Nike Bella Kai
390                      Nike Canyon
482              Nike Classic Cortez
511              Nike Classic Cortez
350              Nike Classic Cortez
323             Nike Cortez '72 S.D.
465                    Nike Daybreak
611                    Nike Daybreak
40                     Nike Daybreak
96                  Nike Daybreak SP
578                   Nike Drop-Type
293               Nike Drop-Type Mid
346           Nike Drop-Type Premium
318                  Nike EXP-X14 SE
328                  Nike Fly.By Mid
279               Nike Huarache Type
32               Nike In-Season TR 9
154                 Nike Kawa Shower
402                 Nike Kawa Shower
168                   Nike M2K Tekno
459                   Nike M2K Tekno
601                   Nike M2K Tekno
28                     Nike Offcourt
477                 Nike Offcourt SE
191                      Nike P-6000
6

The 47 products above were categorized as "Other" because no single keyword captured more than 3 products under a meaningful shared label. Every named shoe line contains at least 4 products.

In [58]:
# Adjust the numbers in pricing columns to be appropriate decimal for USD
df[['listing_price','sale_price']]= df[['listing_price','sale_price']]/100.0
print(df[['listing_price', 'sale_price']])

     listing_price  sale_price
0             0.00       74.95
1             0.00       74.95
2             0.00       99.95
3             0.00       99.95
4             0.00       99.95
..             ...         ...
635           0.00       64.95
637           0.00      139.95
638         159.95      127.97
641           0.00      169.95
642          89.95       62.97

[555 rows x 2 columns]


`listing_price` and `sale_price` were stored as integers in hundredths of a dollar (e.g., 7495 = $74.95). Both columns divided by 100.0 and converted to float.

In [59]:
# Create bins for pricing categories
def get_price_tier(price):
    if price   < 60.00:  return 'Budget'
    elif price < 120.00: return 'Mid'
    elif price < 200.00: return 'Premium'
    else:                return 'Luxury'

df['price_tier'] = df['sale_price'].apply(get_price_tier)
print(df['price_tier'].value_counts())

price_tier
Mid        319
Premium    140
Budget      86
Luxury      10
Name: count, dtype: int64


Products binned into four price tiers based on `sale_price`: Budget (< $60), Mid ($60–$119.99), Premium ($120–$199.99), Luxury ($200+). The majority of the catalog (57.5%) falls in the Mid tier. Only 10 products qualify as Luxury.

In [60]:
print(df.shape)

(555, 11)


In [61]:
print(df.dtypes)

product_name      object
product_id        object
listing_price    float64
sale_price       float64
brand             object
rating           float64
reviews            int64
is_rated            bool
has_msrp            bool
shoe_line         object
price_tier        object
dtype: object


In [62]:
print(df.isnull().sum())

product_name     0
product_id       0
listing_price    0
sale_price       0
brand            0
rating           0
reviews          0
is_rated         0
has_msrp         0
shoe_line        0
price_tier       0
dtype: int64


In [63]:
print(df.head())

                     product_name  product_id  listing_price  sale_price  \
0  Nike Air Force 1 '07 Essential  CJ1646-600            0.0       74.95   
1            Nike Air Force 1 '07  CT4328-101            0.0       74.95   
2    Nike Air Force 1 Sage Low LX  CI3482-200            0.0       99.95   
3             Nike Air Max Dia SE  CD0479-200            0.0       99.95   
4             Nike Air Max Verona  CZ6156-101            0.0       99.95   

  brand  rating  reviews  is_rated  has_msrp  shoe_line price_tier  
0  Nike     0.0        0     False     False  Air Force        Mid  
1  Nike     0.0        0     False     False  Air Force        Mid  
2  Nike     0.0        0     False     False  Air Force        Mid  
3  Nike     0.0        0     False     False    Air Max        Mid  
4  Nike     0.0        0     False     False    Air Max        Mid  


In [64]:
# df subsets to help with EDA
df_rated = df[df['is_rated'] == True].copy()
df_msrp = df[df['has_msrp'] == True].copy()

In [65]:
df.describe()

,listing_price,sale_price,rating,reviews
count,555.000000,555.000000,555.000000,555.000000
mean,41.157387,102.501550,2.794955,7.493694
std,60.815069,42.638028,2.121859,16.714027
min,0.000000,15.950000,0.000000,0.000000
25%,0.000000,71.970000,0.000000,0.000000
50%,0.000000,99.950000,3.900000,2.000000
75%,89.950000,128.960000,4.600000,7.000000
max,199.950000,365.000000,5.000000,223.000000


In [66]:
df_rated.describe()

,listing_price,sale_price,rating,reviews
count,365.000000,365.000000,365.000000,365.000000
mean,46.027260,103.344575,4.249863,11.394521
std,62.330186,40.552337,0.807609,19.509169
min,0.000000,15.950000,1.000000,1.000000
25%,0.000000,72.950000,3.900000,2.000000
50%,0.000000,99.950000,4.400000,5.000000
75%,99.950000,129.950000,5.000000,13.000000
max,189.950000,249.950000,5.000000,223.000000


In [67]:
df_msrp.describe()

,listing_price,sale_price,rating,reviews
count,193.000000,193.000000,193.000000,193.000000
mean,118.354145,84.597979,3.013990,6.911917
std,38.568019,29.654385,1.979547,10.803043
min,29.950000,20.970000,0.000000,0.000000
25%,89.950000,62.970000,0.000000,0.000000
50%,119.950000,79.970000,4.000000,3.000000
75%,159.950000,111.970000,4.500000,9.000000
max,199.950000,159.970000,5.000000,72.000000


## Cleaning Summary
- **Raw records:** 643
- **Duplicates removed:** 88 (13.7%)
- **Columns dropped:** `discount` (zero variance), `description`, `images` (no analytical value)
- **Columns added:** `is_rated`, `has_msrp`, `shoe_line`, `price_tier`
- **Final dataset:** 555 records, 11 columns, 0 nulls

---

## Phase 2: Analysis (SQLite)

In [72]:
import sqlite3

# Create in-memory database
conn = sqlite3.connect(':memory:')

# Load df into SQLite
df.to_sql(
    'shoes',
    conn,
    if_exists='replace',
    index=False
)

print("Table loaded successfully.")

Table loaded successfully.


# SQL Question 1
    1. Which shoe lines have the highest average sale price?


In [73]:
query = """
    SELECT
        shoe_line,
        ROUND(AVG(sale_price),2) AS avg_sale_price
    FROM
        shoes
    GROUP BY
        shoe_line
    ORDER BY
        avg_sale_price DESC
"""

pd.read_sql(query,conn)

,shoe_line,avg_sale_price
0,Flyknit,182.47
1,VaporMax,153.06
2,MX,150.35
3,Air Max,119.02
4,Jordan,118.93
5,Pegasus,116.44
6,Joyride,114.85
7,NBA,110.92
8,Zoom,108.09
9,Metcon,106.11


Flyknit (182.47), VaporMax (153.06), and MX (150.35) command the highest average sale prices — all performance or premium lifestyle lines. Budget-oriented lines like Benassi (27.96) and Court (44.04) anchor the low end. Note that Flyknit and MX have very small catalog presence (6 and 5 products respectively), so their averages reflect a narrow, high-price range rather than a broad market position.

# SQL Question 2
    2. Which shoe lines have the highest average rating (rated products only)?


In [74]:
# Load in df with only rated products into database as new table

df_rated.to_sql(
    'rated_shoes',
    conn,
    if_exists='replace',
    index=False
)

print("Table loaded successfully.")

Table loaded successfully.


In [77]:
# Query the new table
query = """
    SELECT
        shoe_line,
        ROUND(AVG(rating),2) AS avg_rating
    FROM
        rated_shoes
    GROUP BY
        shoe_line
    ORDER BY
        avg_rating DESC;
"""

pd.read_sql(query,conn)

,shoe_line,avg_rating
0,Renew,4.70
1,SB,4.67
2,Mercurial,4.63
3,Air Force,4.56
4,Jordan,4.48
5,Joyride,4.43
6,NBA,4.42
7,Other,4.36
8,Air Max,4.32
9,React,4.30


Among rated products only, Renew (4.70), SB (4.67), and Mercurial (4.63) lead in average rating — but Renew has only 4 total products, making its average less reliable. Tiempo (2.85) is a significant outlier at the bottom, nearly a full point below the next lowest line. Interestingly, Flyknit and VaporMax — the two highest-priced lines — rank 15th and 19th in rating, suggesting price does not drive consumer satisfaction.

# SQL Question 3
3. What is the relationship between price teir and average rating?


In [90]:
query = """
    SELECT
        price_tier,
        ROUND(AVG(rating),2) AS avg_rating
    FROM
        rated_shoes
    GROUP BY
        price_tier
    ORDER BY CASE price_tier
        WHEN 'Budget'  THEN 1
        WHEN 'Mid'     THEN 2
        WHEN 'Premium' THEN 3
        WHEN 'Luxury'  THEN 4
    END
"""

pd.read_sql(query,conn)

,price_tier,avg_rating
0,Budget,4.09
1,Mid,4.29
2,Premium,4.23
3,Luxury,4.18


Rating differences across price tiers are narrow (4.09–4.29), suggesting price point has little relationship with consumer satisfaction among rated products. Mid-tier shoes score highest despite not being the most expensive, which may reflect the largest and most diverse sample size (319 products). Luxury's sample of 10 products is too small to draw firm conclusions.

# SQL Question 4
    4. Which individual products have the most reviews?


In [87]:
query = """
    SELECT
        product_name,
        product_id,
        reviews
    FROM
        shoes
    ORDER BY
        reviews DESC
    LIMIT 10;
"""

pd.read_sql(query, conn)

,product_name,product_id,reviews
0,Air Jordan 10 Retro,310805-137,223
1,Nike Zoom Fly,880848-005,105
2,Nike Air Monarch IV,415445-102,94
3,Nike Air Max 270,AH8050-100,81
4,Nike Air Force 1 '07,315122-001,78
5,Nike Epic React Flyknit 2,BQ8928-011,72
6,Nike Air Max 2017,849559-004,71
7,Nike React Infinity Run Flyknit,CD4371-001,68
8,Nike Air Force 1 '07,315115-112,67
9,Nike Joyride Run Flyknit,AQ2730-009,65


Air Jordan 10 Retro leads with 223 reviews — more than double the second-place Nike Zoom Fly (105). Rows 4 and 8 both show "Nike Air Force 1 '07" but with different product IDs (315122-001 and 315115-112), confirming these are distinct colorways, not duplicates. The top 10 spans Jordan, Zoom, Air Max, Air Force, React, and Joyride lines, indicating review volume is spread across categories rather than concentrated in one.

# SQL Question 5
5. How does the presence vs. absence of a listing price affect average rating and reviews?


In [92]:
query = """
    SELECT
        'All Products' AS scope,
        has_msrp,
        COUNT(*)  AS total_products,
        ROUND(AVG(rating),2) AS avg_rating,
        ROUND(AVG(reviews),2) AS avg_reviews
    FROM
        shoes
    GROUP BY
        has_msrp

    UNION ALL

    SELECT
        'Rated Only' AS scope,
        has_msrp,
        COUNT(*) AS total_products,
        ROUND(AVG(rating),2) AS avg_rating,
        ROUND(AVG(reviews),2) AS avg_reviews
    FROM
        shoes
    WHERE
        is_rated = 1
    GROUP BY
        has_msrp;
"""

pd.read_sql(query,conn)

,scope,has_msrp,total_products,avg_rating,avg_reviews
0,All Products,0,362,2.68,7.80
1,All Products,1,193,3.01,6.91
2,Rated Only,0,224,4.33,12.61
3,Rated Only,1,141,4.13,9.46


When including all products, items without an MSRP average more reviews (7.80 vs. 6.91) but lower ratings (2.68 vs. 3.01) — largely because the no-MSRP group contains more unrated products. Filtering to rated products only tells a cleaner story: no-MSRP items still average more reviews (12.61 vs. 9.46) despite slightly higher ratings (4.33 vs. 4.13). Products without a listing price may see higher review volume because they represent full-price standard catalog items purchased more broadly.

# SQL Question 6
 6. What percentage of products fall into each shoe line?


In [85]:
query = """
    SELECT
        shoe_line,
        ROUND(COUNT(*) *100.0 / (SELECT COUNT(*) FROM shoes),2) AS perc_of_total_shoes
    FROM 
        shoes
    GROUP BY
        shoe_line
    ORDER BY
        perc_of_total_shoes DESC
"""

pd.read_sql(query,conn)

,shoe_line,perc_of_total_shoes
0,Air Max,24.32
1,Zoom,9.37
2,Other,8.47
3,React,7.75
4,Jordan,7.03
5,Mercurial,5.95
6,Air Force,5.23
7,NBA,4.68
8,Pegasus,4.14
9,Phantom,2.52


Air Max accounts for nearly 1 in 4 products in the catalog (24.32%), making it by far the dominant line. The top 5 lines — Air Max, Zoom, Other, React, and Jordan — together represent 57% of the catalog. Fifteen of the 24 shoe lines each represent less than 2% of products, indicating a long tail of niche or limited-run lines.

In [93]:
df.to_csv(r'C:\Users\JaceCordell\Desktop\nike_shoe_analysis\1_data\clean\clean_nike_shoe_sales.csv')